[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance2_exercices.ipynb)

# Séance 2.2 — Nettoyer des données réelles

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les sept défauts classiques d'un fichier réel
- convertir du texte en nombres et en dates
- traiter les valeurs manquantes en connaissance de cause
- supprimer les doublons et écarter les valeurs aberrantes
- construire un pipeline de nettoyage qu'on peut rejouer

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")

print(sale.shape)
sale.head(3)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Dédoublonner (toujours en premier)

> **Votre mission :**
> - Créer `net` : `sale` sans les lignes dupliquées.
> - ⚠️ On dédoublonne **avant tout le reste**. Filtrer d'abord ferait perdre le compte de ce qu'on retire.

In [ ]:
net = sale.____().copy()

print(len(net))

In [ ]:
verifier("1 - apres dedoublonnage", len(net) == 5119,
         "5370 - 251 : la methode s'appelle drop_duplicates()")

### Exercice 2 — Écarter les ventes sans client

> **Votre mission :**
> - Retirer de `net` les lignes dont `client_id` est manquant.
> - Rappel : on le fait parce que notre question porte sur les **clients**. Pour une question sur le chiffre d'affaires total, ce serait une erreur.

In [ ]:
net = net.____(subset=["client_id"]).copy()

print(len(net))

In [ ]:
verifier("2 - lignes avec client", len(net) == 4712,
         "dropna(subset=[...]) cible une colonne precise")

### Exercice 3 — Le prix en nombre

> **Votre mission :**
> - Enlever le suffixe ` EUR`, remplacer la virgule par un point, convertir en nombre.
> - Remettre le résultat dans `net["prix"]`.
> - Puis vérifier qu'aucune valeur n'a été perdue : `nb_prix_perdus`.

In [ ]:
txt = net["prix"].str.replace(" EUR", "", regex=False)
txt = txt.str.replace(",", "____", regex=False)

net["prix"] = pd.____(txt, errors="coerce")
nb_prix_perdus = net["prix"].isna().sum()

print(net["prix"].dtype, "|", nb_prix_perdus, "valeurs perdues")

In [ ]:
verifier("3a - prix numerique", net["prix"].dtype == "float64",
         "pd.to_numeric convertit une colonne texte en nombres")
verifier("3b - aucune perte", nb_prix_perdus == 0,
         "si > 0, c'est qu'il reste du texte non converti dans la colonne")

### Exercice 4 — Les dates, correctement

> **Votre mission :**
> - Convertir `net["date"]` en vraies dates.
> - Le fichier mélange `14/11/2011` et `24-11-2011`, et il est au format **français**.
> - Puis mettre la date la plus ancienne dans `date_min`.

In [ ]:
net["date"] = pd.to_datetime(net["date"], format="____", dayfirst=____)
date_min = net["date"].min()

print(date_min)

In [ ]:
verifier("4 - date la plus ancienne", str(date_min)[:10] == "2010-12-01",
         "sans dayfirst=True, 01/12/2010 est lu comme le 12 janvier")

### Exercice 5 — Extraire le mois

> **Votre mission :**
> - Créer la colonne `mois` à partir de `date`.
> - Mettre le numéro du mois qui compte le plus de lignes dans `mois_top`.

In [ ]:
net["mois"] = net["date"].____.month
mois_top = net["mois"].value_counts().____()

print(mois_top)

In [ ]:
verifier("5 - mois le plus charge", mois_top == 10,
         ".dt.month sur une colonne de dates, puis value_counts().idxmax()")

### Exercice 6 — Uniformiser les catégories

> **Votre mission :**
> - Enlever les espaces autour et tout passer en minuscules.
> - Mettre le nombre de catégories restantes dans `nb_cat`.

In [ ]:
net["categorie"] = net["categorie"].str.____().str.____()
nb_cat = net["categorie"].nunique()

print(nb_cat)

In [ ]:
verifier("6 - categories uniformisees", nb_cat == 8,
         "il en reste beaucoup plus si vous n'avez fait que l'une des deux operations")

### Exercice 7 — Retours et aberrations

> **Votre mission :**
> - Compter les **retours** (`qte` strictement négatif) dans `nb_retours`.
> - Compter les quantités **aberrantes** (`qte` ≥ 10 000) dans `nb_aberrants`.
> - Puis créer `final` : les quantités strictement positives et inférieures à 10 000.

In [ ]:
nb_retours = len(net.query("____"))
nb_aberrants = len(net.query("qte >= 10000"))

final = net.query("qte > 0 ____ qte < 10000").copy()

print(nb_retours, "retours |", nb_aberrants, "aberrants |", len(final), "conservees")

In [ ]:
verifier("7a - retours", nb_retours == 107, "la condition est qte < 0")
verifier("7b - aberrants", nb_aberrants == 14, "la condition est qte >= 10000")
verifier("7c - lignes conservees", len(final) == 4591,
         "dans query() les conditions se combinent avec and")

### Exercice 8 — Le compte rendu

> **Votre mission :**
> - Calculer le **taux de perte** en pourcentage, arrondi à 1 décimale, dans `taux_perte`.
> - Formule : `100 * (1 - lignes_finales / lignes_initiales)`.
> - C'est le chiffre que vous présentez à votre responsable — pas « j'ai nettoyé les données ».

In [ ]:
taux_perte = round(100 * (1 - len(____) / len(____)), 1)

print("perte :", taux_perte, "%")

In [ ]:
verifier("8 - taux de perte", taux_perte == 14.5,
         "comparez le fichier final au fichier de depart sale")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Le diagnostic, en pourcentage

> **Votre mission :**
> - Calculer le **taux de valeurs manquantes de chaque colonne**, en %, arrondi à 2 décimales.
> - Combien de colonnes sont réellement touchées ?
> - *Nouveau :* sur des True/False, `.mean()` donne directement une proportion — `df.isna().mean()`.

### Question 10 — Le prix d'un `dropna()` négligent

> **Votre mission :**
> - Combien de lignes resteraient après un `dropna()` **sans argument** ?
> - Et après `dropna(subset=['client_id'])` ?
> - Ici les deux donnent le même résultat. Dans quel cas seraient-ils très différents ?

### Question 11 — Le doublon qui n'en a pas l'air

> **Votre mission :**
> - Compter les doublons **exacts**, puis les doublons sur le seul couple `cmd_id` + `prod_id`.
> - Les deux nombres diffèrent. Afficher les lignes concernées et expliquer ce qui s'est passé.
> - *Nouveau :* `df.duplicated(subset=['a', 'b'])` ne compare que les colonnes nommées.

### Question 12 — Regarder avant de convertir

> **Votre mission :**
> - Convertir `prix` en nombre avec `errors='coerce'`, **sans écraser la colonne d'origine**.
> - Combien de valeurs échouent ? **Afficher quelques-unes des valeurs fautives d'origine.**
> - Ne jamais lancer un `coerce` sans avoir regardé ce qu'on s'apprête à transformer en `NaN`.

### Question 13 — La fenêtre d'observation

> **Votre mission :**
> - Convertir `date` correctement, puis donner la date la plus ancienne, la plus récente, et le nombre de **jours** entre les deux.
> - Un rapport annuel calculé sur cette période serait-il honnête ?
> - *Nouveau :* une soustraction de dates donne une durée ; `.days` en extrait le nombre de jours.

### Question 14 — Les aberrations, sans seuil arbitraire

> **Votre mission :**
> - En partie 1, on a écarté `qte >= 10000` — un seuil choisi à la main.
> - Refaire le repérage avec la **règle de l'écart interquartile** : est aberrant ce qui dépasse `q3 + 1.5 * (q3 - q1)`.
> - Combien de lignes dépasse-t-elle ? Faut-il toutes les supprimer ?

### Question 15 — Le compte rendu qualité

> **Votre mission :**
> - Vous rendez le fichier nettoyé. Produire les **quatre chiffres** de la note d'accompagnement :
> - lignes au départ · lignes conservées · taux de perte en % · **part du chiffre d'affaires réel** que représentent les ventes écartées faute de client identifié.
> - Ce dernier chiffre est celui qu'on oublie. Comparez-le au taux de perte en lignes : que vous dit l'écart entre les deux ?
> - ⚠️ Calculez ce CA sur les ventes **plausibles** uniquement. Les quantités à 99 999 produiraient sinon un total fictif qui écrase tout le reste.